In [1]:
import gdown
import tempfile
import os
import cv2
import zipfile
import numpy as np
from tqdm import tqdm
import time
import subprocess
from PIL import Image
from ultralytics import YOLO

In [2]:
global subprocess

In [3]:
url = "https://drive.google.com/uc?id=1TMEd1MA0Vy4PJycfBjx2a_EG0i9jpOzN"

In [4]:
script_path = "./training data/detect.py"
weights_file = "./training data/yolov5s.pt"
video_path = "../../../../../../Downloads/2-Jeu_Test_J2-10h30_P5 11.17.01.mp4"
output_folder = "frame_results"

In [5]:
import traceback

def execute_detect_command(frame_buffer):
    # Charger le modèle YOLOv5
    model = YOLO(weights_file)
    
    try:
        # Préparer l'image
        img = frame_buffer[0]
        
        # Faire la détection
        results = model(img)
        
        # Récupérer les véhicules détectés
        vehicle_boxes = []
        
        for result in results:
            for detection in result:
                confidence = float(detection[4])
                class_id = int(detection[5])
                x, y, w, h = detection[0:4]
                
                # Filtrer les véhicules (vous pouvez ajuster ces critères)
                if confidence > 0.5 and class_id in [2, 7]:  # Classe 0 pour voiture, classe 2 pour camion
                    vehicle_boxes.append((x, y, w, h, confidence, class_id))
                    
        print(f"{len(vehicle_boxes)} véhicules détectés dans cette frame")
        return vehicle_boxes
    
    except Exception as e:
        print(f"Erreur lors du traitement de la frame dans la fonction : {str(e)}")
        traceback.print_exc()
        return []

In [6]:
classes = []
with open("./training data/coco.names", "r") as f:
    classes = [line.strip() for line in f.readlines()]

# Créez une liste des classes de véhicules
vehicle_classes = [cls for cls in classes if cls.startswith(("car", "truck"))]

confidence_threshold = 0.5
classes[2], classes[7]

('car', 'truck')

In [7]:
def reduce_resolution(image):
    height, width = image.shape[:2]
    ratio = min(640 / height, 640 / width)
    new_height = int(height * ratio)
    new_width = int(width * ratio)
    
    # Assurez-vous que la nouvelle taille est au moins 416x416
    new_height = max(new_height, 416)
    new_width = max(new_width, 416)
    
    # Normalise l'image avant la réduction de résolution
    normalized_image = image.astype(np.float32) / 255.0
    
    # Appliquez la réduction de résolution sur l'image normalisée
    resized_image = cv2.resize(normalized_image, (int(new_width), int(new_height)))
    
    # Dénormalisez l'image après la réduction de résolution
    resized_image = (resized_image * 255).astype(np.uint8)
    
    return resized_image


In [8]:
def blob_from_image(image):
    mean = np.array([104., 117., 123.])  # Moyenne de BGR
    return cv2.dnn.blobFromImage(image, 1/255, (416, 416), mean, swapRB=True, crop=False)


In [9]:
def process_single_frame(frame_buffer):
    #preprocessed_frame = preprocess_image(frame_buffer[0])
    vehicle_boxes = detect_and_segment_vehicles(frame_buffer[0])#preprocessed_frame)
    return vehicle_boxes


In [10]:
def preprocess_image(image):
    # Nettoie les images (filtrage, correction de luminosité)
    lab_image = cv2.cvtColor(image, cv2.COLOR_BGR2LAB)
    
    # Corrige de luminosité
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
    lab_image[:,:,0] = clahe.apply(lab_image[:,:,0])
    
    # Retourne à l'espace BGR sans modification
    return image  # Retourne l'image originale en BGR


In [11]:
def draw_bounding_boxes(frame, vehicle_boxes):
    detected_frame = frame.copy()
    for box in vehicle_boxes:
        x, y, w, h = box
        color = tuple(np.random.randint(0, 256, 3).tolist())
        cv2.rectangle(detected_frame, (x, y), (x+w, y+h), color, 2)
    return detected_frame


In [12]:
def display_results(results):
    for result in results:
        cv2.imshow("Frame with detections", result["detected_image"])
        cv2.waitKey(0)
        cv2.destroyAllWindows()

In [13]:
import matplotlib.pyplot as plt

def show_results(results):
    fig, ax = plt.subplots(figsize=(20, 10))
    for result in results:
        ax.imshow(result["detected_image"])
        ax.set_title(f'Frame {result["frame_number"]}, Véhicules détectés: {len(result["vehicle_boxes"])}')
        plt.axis('off')
        plt.show(block=False)
        plt.pause(0.5)
        plt.close()


In [14]:
import cv2
import matplotlib.pyplot as plt
from IPython.display import clear_output

def display_results(results, show_matplotlib=False):
    for i, result in enumerate(results):
        # Afficher avec OpenCV
        img_with_boxes = draw_bounding_boxes(result["original_image"], result["vehicle_boxes"])
        cv2.imshow("Frame with detections", img_with_boxes)
        cv2.waitKey(25)  # Attendre 25ms pour une animation lente
        
        # Afficher avec Matplotlib (optionnel)
        if show_matplotlib:
            clear_output(wait=True)
            plt.figure(figsize=(10, 6))
            plt.imshow(img_with_boxes)
            plt.title(f'Frame {result["frame_number"]}, Véhicules détectés: {len(result["vehicle_boxes"])}')
            plt.axis('off')
            plt.show(block=False)
            plt.pause(0.5)
        
        cv2.destroyAllWindows()  # Fermer les fenêtres OpenCV après chaque frame

def draw_bounding_boxes(image, boxes):
    img_copy = image.copy()
    for box in boxes:
        x, y, w, h, confidence, class_id = box
        color = (0, 255, 0) if class_id == 0 else (0, 0, 255)  # Vert pour voiture, Rouge pour camion
        cv2.rectangle(img_copy, (x, y), (x+w, y+h), color, 2)
        cv2.putText(img_copy, f"C{class_id} {confidence:.2f}", (x, y-10),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.9, color, 2)
    
    return img_copy

In [15]:
def main():
    
    segment_duration = 20  # Durée de chaque segment en secondes
    
    try:
        # Télécharge le fichier dans un dossier temporaire
        temp_dir = tempfile.TemporaryDirectory()
        output_path = os.path.join(temp_dir.name, "2-Jeu_Test_J2-10h30_P5 11.17.01.mp4.zip")

        print("Téléchargement...")
        gdown.download(url, output=output_path)
        print(f"Fichier téléchargé dans : {output_path}")

        # Extrait tous les fichiers MP4 du ZIP
        mp4_files = []
        with zipfile.ZipFile(output_path, 'r') as zip_ref:
            mp4_files = [member.filename for member in zip_ref.infolist() if member.filename.endswith('.mp4')]

        if len(mp4_files) == 0:
            raise ValueError("Aucun fichier MP4 trouvé dans le ZIP.")
        elif len(mp4_files) > 1:
            print(f"Plusieurs fichiers MP4 trouvés : {', '.join(mp4_files)}")
            # Choisit le premier fichier MP4 pour l'extraction
            mp4_filename = mp4_files[0]
        else:
            mp4_filename = mp4_files[0]

        extracted_mp4_path = os.path.join(temp_dir.name, mp4_filename)

        # Extrait le fichier MP4 sélectionné en excluant les fichiers cachés Mac OS
        with zipfile.ZipFile(output_path, 'r') as zip_ref:
            mp4_file_info = next(filter(lambda m: m.filename == mp4_filename, zip_ref.infolist()))
            zip_ref.extract(member=zip_ref.infolist()[zip_ref.infolist().index(next(filter(lambda m: m.filename == mp4_filename, zip_ref.infolist())))], path=temp_dir.name, pwd=b"password")       
            print(f"Fichier MP4 extrait dans : {extracted_mp4_path}")

        # Charge et lire la vidéo
        video = cv2.VideoCapture(extracted_mp4_path)

        if not video.isOpened():
            raise Exception("Impossible d'ouvrir la vidéo.")
            
        fps = video.get(cv2.CAP_PROP_FPS)
        frame_count = int(video.get(cv2.CAP_PROP_FRAME_COUNT))
        duration = frame_count / fps

        print(f"Vidéo prête à l'utilisation. Durée totale : {duration:.2f}s")

        # Lit une frame pour vérifier le contenu
        #ret, frame = video.read()
        #if ret:
            #print(f"Première frame lue avec succès. Taille : {frame.shape}")
        #else:
            #raise Exception("Impossible de lire la première frame de la vidéo.")

        #if not video.isOpened():
            #print("Error opening video stream or file")
            #exit()

        width = int(video.get(cv2.CAP_PROP_FRAME_WIDTH))
        height = int(video.get(cv2.CAP_PROP_FRAME_HEIGHT))
        fps = int(video.get(cv2.CAP_PROP_FPS))

        print(f"Video info: {width}x{height}, {fps} fps")

        # Initialize the video writer
        fourcc = cv2.VideoWriter_fourcc(*'mp4v')
        out = cv2.VideoWriter('output.mp4', fourcc, fps, (width, height))

        # Boucle principale pour traiter chaque batch de frames
        frame_number= 0
        results =[]

        start_time = time.time()
        last_frame_time = start_time      
                
        #if not os.path.exists(yolov3_weights_path) or not os.path.exists(yolov3_cfg_path):
            #raise FileNotFoundError("Le fichier YOLO poids ou configuration manque.")

        #try:
            #with open(yolov3_cfg_path, "rb") as f:
                #config_data = f.read()
            #print(config_data[:50])  # Affiche les premières lignes du fichier pour vérifier son contenu
        #except Exception as e:
            #print(f"Erreur lors de la lecture du fichier de configuration YOLO : {str(e)}")

        batch_size = 60  # Nombre de frames par batch
        total_batches_per_segment = int(segment_duration * fps / batch_size)

        with tqdm(total=int(video.get(cv2.CAP_PROP_FRAME_COUNT))) as pbar:
            frame_number = 0
            frame_buffer = []
            total_frames_read = 0
            while True:
                try:
                    start_frame = frame_number
                    end_frame = start_frame + segment_duration * fps
                    
                    # Nettoyez le buffer avant d'ajouter de nouvelles frames
                    frame_buffer.clear()
                    
                    for _ in range(batch_size):
                        ret, frame = video.read()
                        if not ret:
                            break
                        total_frames_read += 1
                        #reduced_frame = reduce_resolution(frame)
                        #frame_buffer.append(reduced_frame)
                        frame_buffer.append(frame)
                    print(f"Total frames read : {total_frames_read}")
                    
                    if not frame_buffer:
                        break
                    
                    if len(frame_buffer) == batch_size:
                        vehicle_boxes_list = execute_detect_command(frame_buffer)
                    else:
                        print(f"Erreur : Le nombre de frames dans le buffer ({len(frame_buffer)}) n'est pas égal à {batch_size}")
                        continue
                        
                    
                    if not vehicle_boxes_list:
                        print(f"Aucun véhicule détecté dans la frame {frame_number}.")
                        continue
                    else:
                        print(f"{len(vehicle_boxes_list)} véhicules détectés dans la frame {frame_number}")
                        #show_results(results)
                        continue

                    # Dessinez les rectangles sur la première image du buffer
                    detected_frame = frame_buffer[0].copy()
                    for box in vehicle_boxes_list:
                        cv2.rectangle(detected_frame, (int(box[0]), int(box[1])), (int(box[2]), int(box[3])), (0,255,0), 2)

                    # Stocke les résultats
                    result_entry = {
                        "frame_number": frame_number,
                        "vehicle_boxes": vehicle_boxes_list[0],
                        "preprocessed_image": preprocess_image(frame_buffer[0]),
                        "detected_image": draw_bounding_boxes(frame_buffer[0], vehicle_boxes_list),
                        "elapsed_time": elapsed_time
                    }
                    results.append(result_entry)
                    
                    print("Résultats du modèle:", results)
                    print("Véhicules détectés:", vehicle_boxes_list)


                    pbar.update(batch_size)

                    # Incrémentation du frame_number après avoir traité toutes les frames dans le buffer
                    frame_number += batch_size

                    time.sleep(2)

                except Exception as e:
                    print(f"Erreur lors du traitement de la frame {frame_number} : {str(e)}")

                current_time = time.time()
                elapsed_time = current_time - last_frame_time
                last_frame_time = current_time
                
                if frame_number >= frame_count:
                    break

        print("Traitement des images et extraction des frames terminés.")
        
        # Ferme la vidéo
        video.release()
        cv2.destroyAllWindows()

    except zipfile.BadZipFile:
        print("Le fichier ZIP est corrompu ou invalide.")
    except Exception as e:
        print(f"Une erreur s'est produite : {str(e)}")

    finally:
        # On s'assure que tous les fichiers temporaires sont supprimés
        if 'temp_dir' in locals():
            temp_dir.cleanup()
        if os.path.exists(output_path):
            os.remove(output_path)
        if os.path.exists(extracted_mp4_path):
            os.remove(extracted_mp4_path)
        print("Tous les fichiers temporaires ont été supprimés.")


In [17]:
net = None
last_frame_time = time.time()
results = []

if __name__ == '__main__':
    main()

Téléchargement...


Downloading...
From (original): https://drive.google.com/uc?id=1TMEd1MA0Vy4PJycfBjx2a_EG0i9jpOzN
From (redirected): https://drive.google.com/uc?id=1TMEd1MA0Vy4PJycfBjx2a_EG0i9jpOzN&confirm=t&uuid=0106a076-4149-4ba0-96de-161f3c7ad8ad
To: /var/folders/l6/g437mjmj6jd2n_lsl9mw8kym0000gn/T/tmpb8n71803/2-Jeu_Test_J2-10h30_P5 11.17.01.mp4.zip
100%|████████████████████████████████████████| 294M/294M [02:04<00:00, 2.35MB/s]


Fichier téléchargé dans : /var/folders/l6/g437mjmj6jd2n_lsl9mw8kym0000gn/T/tmpb8n71803/2-Jeu_Test_J2-10h30_P5 11.17.01.mp4.zip
Plusieurs fichiers MP4 trouvés : 2-Jeu_Test_J2-10h30_P5 11.17.01.mp4, __MACOSX/._2-Jeu_Test_J2-10h30_P5 11.17.01.mp4
Fichier MP4 extrait dans : /var/folders/l6/g437mjmj6jd2n_lsl9mw8kym0000gn/T/tmpb8n71803/2-Jeu_Test_J2-10h30_P5 11.17.01.mp4
Vidéo prête à l'utilisation. Durée totale : 116.97s
Video info: 1920x1080, 30 fps


  0%|                                                  | 0/3509 [00:00<?, ?it/s]

Total frames read : 60
PRO TIP 💡 Replace 'model=./training data/yolov5s.pt' with new 'model=./training data/yolov5su.pt'.
YOLOv5 'u' models are trained with https://github.com/ultralytics/ultralytics and feature improved performance vs standard YOLOv5 models trained with https://github.com/ultralytics/yolov5.


0: 384x640 9 cars, 126.6ms
Speed: 5.1ms preprocess, 126.6ms inference, 9.2ms postprocess per image at shape (1, 3, 384, 640)
Erreur lors du traitement de la frame dans la fonction : index 4 is out of bounds for dimension 0 with size 1
Aucun véhicule détecté dans la frame 0.
Total frames read : 120
PRO TIP 💡 Replace 'model=./training data/yolov5s.pt' with new 'model=./training data/yolov5su.pt'.
YOLOv5 'u' models are trained with https://github.com/ultralytics/ultralytics and feature improved performance vs standard YOLOv5 models trained with https://github.com/ultralytics/yolov5.



...

Traceback (most recent call last):
  File "/var/folders/l6/g437mjmj6jd2n_lsl9mw8kym0000gn/T/ipykernel_50979/1413121527.py", line 19, in execute_detect_command
    confidence = float(detection[4])
  File "/Users/davidr/.pyenv/versions/3.10.6/envs/lewagon_current/lib/python3.10/site-packages/ultralytics/engine/results.py", line 288, in __getitem__
    return self._apply("__getitem__", idx)
  File "/Users/davidr/.pyenv/versions/3.10.6/envs/lewagon_current/lib/python3.10/site-packages/ultralytics/engine/results.py", line 360, in _apply
    setattr(r, k, getattr(v, fn)(*args, **kwargs))
  File "/Users/davidr/.pyenv/versions/3.10.6/envs/lewagon_current/lib/python3.10/site-packages/ultralytics/engine/results.py", line 184, in __getitem__
    return self.__class__(self.data[idx], self.orig_shape)
IndexError: index 4 is out of bounds for dimension 0 with size 1



0: 384x640 13 cars, 1 truck, 74.5ms
Speed: 3.3ms preprocess, 74.5ms inference, 0.6ms postprocess per image at shape (1, 3, 384, 640)
Erreur lors du traitement de la frame dans la fonction : index 4 is out of bounds for dimension 0 with size 1
Aucun véhicule détecté dans la frame 0.


Traceback (most recent call last):
  File "/var/folders/l6/g437mjmj6jd2n_lsl9mw8kym0000gn/T/ipykernel_50979/1413121527.py", line 19, in execute_detect_command
    confidence = float(detection[4])
  File "/Users/davidr/.pyenv/versions/3.10.6/envs/lewagon_current/lib/python3.10/site-packages/ultralytics/engine/results.py", line 288, in __getitem__
    return self._apply("__getitem__", idx)
  File "/Users/davidr/.pyenv/versions/3.10.6/envs/lewagon_current/lib/python3.10/site-packages/ultralytics/engine/results.py", line 360, in _apply
    setattr(r, k, getattr(v, fn)(*args, **kwargs))
  File "/Users/davidr/.pyenv/versions/3.10.6/envs/lewagon_current/lib/python3.10/site-packages/ultralytics/engine/results.py", line 184, in __getitem__
    return self.__class__(self.data[idx], self.orig_shape)
IndexError: index 4 is out of bounds for dimension 0 with size 1
  0%|                                                  | 0/3509 [00:21<?, ?it/s]

Total frames read : 3509
Erreur : Le nombre de frames dans le buffer (29) n'est pas égal à 60
Total frames read : 3509
Traitement des images et extraction des frames terminés.


Tous les fichiers temporaires ont été supprimés.


In [39]:
display_results(results[0], show_matplotlib=True)


IndexError: list index out of range

In [27]:
for result in results:
    for detection in result:
        print("Detection:", detection)
        print("Type de detection:", type(detection))
        print("Nombre d'éléments:", len(detection))
        print("Premiers éléments:", detection[:6])  # Affiche les six premiers éléments
        print()


In [28]:
import cv2

def get_video_fps(video_path):
    cap = cv2.VideoCapture(video_path)
    fps = cap.get(cv2.CAP_PROP_FPS)
    cap.release()
    return fps

video_path = "chemin_vers_votre_video.mp4"
fps = get_video_fps(video_path)
print(f"Nombre de frames par seconde : {fps}")

Nombre de frames par seconde : 0.0


OpenCV: Couldn't read video stream from file "chemin_vers_votre_video.mp4"


In [32]:
plt.imshow(frame_buffer[0])
plt.axis('off')

NameError: name 'frame_buffer' is not defined

In [36]:
cv2.imshow(results[0])

IndexError: list index out of range